# Instalar dependências

In [2]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl

^C


# Login no Hugging Face (necessário para LLaMA)

In [1]:
from huggingface_hub import notebook_login
notebook_login()

# Carregar dataset JSON

In [2]:
import json
import pandas as pd
from datasets import Dataset

data = pd.read_parquet("dataset_finetuning.parquet")

dataset = Dataset.from_pandas(data)
dataset

Dataset({
    features: ['text'],
    num_rows: 665
})

# Formatando dataset para treinamento

In [ ]:
#Usar apenas para o dataset antigo ou não formatados
import json

def format_example(example):
    instruction = "Convert project description into structured GitHub issues JSON."

    input_text = example["input"]
    output_text = json.dumps(example["output"], indent=2)

    formatted = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output_text}"""

    return {"text": formatted}

dataset = dataset.map(format_example)
dataset = dataset.remove_columns(["input", "output"])
dataset

# Carregar modelo com QLoRA (4-bit)

In [3]:
import torch
# Para funcionar em GPU nvidia, python 3.11.9 exatamente
#ver se a GPU esta funcionando (nvidia-smi)- para ver a GPU cuda cors version
# py -3.11 -m install torch torchvision --index-url https://download.pytorch.org/whl/cu130 (para instalar com compatibilidade aos cudas da nvidia)
# py -3.11 -m pip uninstall torch torchvision torchaudio (para se der errado)
# https://pytorch.org/get-started/locally/
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 5070


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# model_name = "meta-llama/Meta-Llama-3-8B"
model_name = "meta-llama/Meta-Llama-3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="cuda", # para usar na cpu, deixar "auto"
    torch_dtype=torch.float16
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

# Configurar LoRA

In [10]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


# Tokenização

In [11]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=2048
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/665 [00:00<?, ? examples/s]

# Treinamento

In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama-backlog",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    learning_rate=2e-4,
    # fp16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/665 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/665 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/665 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss
10,0.941850
20,0.658226
30,0.504603
40,0.488667
50,0.475635
60,0.463064
70,0.457035
80,0.427378
90,0.431095
100,0.414653


TrainOutput(global_step=501, training_loss=0.37148523467505523, metrics={'train_runtime': 12124.3867, 'train_samples_per_second': 0.165, 'train_steps_per_second': 0.041, 'total_flos': 4.353263899803648e+16, 'train_loss': 0.37148523467505523})

# Salvar modelo fine-tuned

In [15]:
model.save_pretrained("llama-backlog-lora")
tokenizer.save_pretrained("llama-backlog-lora")

('llama-backlog-lora\\tokenizer_config.json',
 'llama-backlog-lora\\tokenizer.json')

# Carregar o Modelo (Apenas se não foi treinado na hora)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
# 1. Configurações e modelo base
base_model_name = "meta-llama/Meta-Llama-3-8B"
adapter_model_path = "llama-backlog-lora"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="cuda",
    torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
# 2. CARREGAR OS PESOS GERADOS NO SEU FINETUNING (LORA)
model = PeftModel.from_pretrained(base_model, adapter_model_path)
# 3. Colocar o modelo em modo de avaliação/inferência
model.eval()

# Testar o modelo

In [ ]:
prompt = """### Instruction:
Convert project description into structured GitHub issues JSON(multiple, as many as needed for the especified problem).
Follow the Input language to generate the output.

### Input:
Criar uma plataforma inteligente que utilize IA para processar documentação (como PDFs) e automatizar a conversão de requisitos técnicos e de negócio em issues estruturadas no GitHub. O sistema deve transformar documentos brutos em tarefas organizadas, com títulos claros, descrições técnicas detalhadas, critérios de aceitação e categorização automática (Feature, Bug, Improvement, Technical Task).

A plataforma deve:

* Automatizar a ingestão de dados, convertendo PDFs em issues completas.
* Padronizar a organização do backlog com aplicação automática de labels e segmentações.
* Gerar estimativas de tempo total de produção e sugerir divisão em sprints com base na densidade do backlog.
* Extrair e documentar requisitos funcionais, não-funcionais, regras de negócio e restrições técnicas.
* Identificar inconsistências entre documentos e sugerir melhorias arquiteturais por meio de relatórios executivos sob demanda.
* Reduzir o gap de comunicação entre negócio e desenvolvimento, garantindo que as tarefas reflitam fielmente os objetivos estratégicos da empresa.


### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=2000,
    temperature=0.2
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Both `max_new_tokens` (=2000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Convert project description into structured GitHub issues JSON.

### Input:
Criar uma plataforma inteligente que utilize IA para processar documentação (como PDFs) e automatizar a conversão de requisitos técnicos e de negócio em issues estruturadas no GitHub. O sistema deve transformar documentos brutos em tarefas organizadas, com títulos claros, descrições técnicas detalhadas, critérios de aceitação e categorização automática (Feature, Bug, Improvement, Technical Task).

A plataforma deve:

* Automatizar a ingestão de dados, convertendo PDFs em issues completas.
* Padronizar a organização do backlog com aplicação automática de labels e segmentações.
* Gerar estimativas de tempo total de produção e sugerir divisão em sprints com base na densidade do backlog.
* Extrair e documentar requisitos funcionais, não-funcionais, regras de negócio e restrições técnicas.
* Identificar inconsistências entre documentos e sugerir melhorias arquiteturais por meio de relatórios executi

In [5]:
prompt = """### Instruction:
Convert project description into structured GitHub issues JSON(multiple, as many as needed for the especified problem).
Follow the Input language to generate the output.

### Input:
Desenvolver uma plataforma web de gerenciamento de eventos e apostas em resultados futuros, onde usuários possam criar eventos, apostar em resultados e acompanhar estatísticas em tempo real. O sistema deve permitir cadastro de usuários, criação e moderação de eventos, registro de apostas, controle financeiro e geração de relatórios analíticos.

A aplicação deverá possuir backend com API REST, interface web responsiva e um sistema de moderação para validar conteúdos antes de serem publicados.

O projeto deve contemplar diferentes tipos de issues (Feature, Bug, Improvement, Technical Task) ao transformar os requisitos abaixo em tarefas estruturadas.

A plataforma deve:

Permitir cadastro, login e autenticação de usuários, com recuperação de senha e validação de email.

Permitir que usuários criem eventos futuros (ex.: jogos, eleições, lançamentos de produtos) nos quais outras pessoas possam apostar.

Implementar sistema de apostas com duas ou mais opções de resultado, registrando valor apostado e histórico de apostas do usuário.

Incluir painel de moderação, onde administradores possam aprovar ou rejeitar eventos criados pelos usuários.

Gerenciar carteira virtual, permitindo depósitos, retiradas e visualização de saldo.

Disponibilizar dashboard com estatísticas, como volume total apostado, eventos mais populares e desempenho do usuário.

Garantir segurança e integridade dos dados, incluindo criptografia de senhas, proteção contra ataques comuns e validação de entradas.

Implementar notificações por email ou sistema interno para informar usuários sobre aprovação de eventos, resultados e ganhos.

Além das funcionalidades principais, o sistema deve:

Identificar possíveis bugs ou inconsistências funcionais, como falhas na validação de apostas ou cálculo incorreto de ganhos.

Propor melhorias de experiência do usuário, como filtros de eventos, busca avançada e interface mais intuitiva.

Incluir tarefas técnicas, como configuração de banco de dados, integração com serviços de pagamento, criação de testes automatizados e pipelines de CI/CD.

O resultado esperado é a geração automática de um backlog estruturado em issues do GitHub, contendo:

Título claro da tarefa

Descrição técnica detalhada

Tipo de issue (Feature, Bug, Improvement ou Technical Task)

Critérios de aceitação

Sugestão de prioridade

Estimativa de esforço

Labels apropriadas para organização do projeto.


### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=2000,
    temperature=0.2
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=2000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Convert project description into structured GitHub issues JSON(multiple, as many as needed for the especified problem).
Follow the Input language to generate the output.

### Input:
Desenvolver uma plataforma web de gerenciamento de eventos e apostas em resultados futuros, onde usuários possam criar eventos, apostar em resultados e acompanhar estatísticas em tempo real. O sistema deve permitir cadastro de usuários, criação e moderação de eventos, registro de apostas, controle financeiro e geração de relatórios analíticos.

A aplicação deverá possuir backend com API REST, interface web responsiva e um sistema de moderação para validar conteúdos antes de serem publicados.

O projeto deve contemplar diferentes tipos de issues (Feature, Bug, Improvement, Technical Task) ao transformar os requisitos abaixo em tarefas estruturadas.

A plataforma deve:

Permitir cadastro, login e autenticação de usuários, com recuperação de senha e validação de email.

Permitir que usuários c

# Ver total de parâmetros

In [18]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total de parâmetros: {total_params:,}")

Total de parâmetros: 4,547,416,064
